# To investigate the case of specificity for reliability level classification

### Import

In [2]:
import tensorflow as tf
from keras.models import Model
from keras.layers import Dense, Input
from keras import activations
from keras import backend as K
import pandas as pd
import numpy as np
from scipy.optimize import minimize, differential_evolution
import glob, math, os
from scipy import special
from sklearn.metrics import recall_score

def euclidean_dist(row):
    # Function to calc euclidean distance on every df row 
    euc_dist = math.sqrt(row["U2G_Distance"]**2 - row["Height"]**2)
    return euc_dist

def q_func(x):
    q = 0.5 - 0.5*special.erf(x / np.sqrt(2))
    return q

def friis_calc(P,freq,dist,ple):
    '''
    Friis path loss equation
    P = Tx transmit power
    freq = Signal frequency
    dist = Transmission distance
    ple = Path loss exponent
    '''
    propagation_speed = 299792458
    l = propagation_speed / freq
    h_pl = P * l**2 / (16*math.pi**2)
    P_Rx = h_pl * dist**(-ple)
    return P_Rx

def plos_calc(h_dist, height_tx, height_rx, env='suburban'):
    '''
    % This function implements the LoS probability model from the paper
    % "Blockage Modeling for Inter-layer UAVs Communications in Urban
    % Environments" 
    % param h_dist    : horizontal distance between Tx and Rx (m)
    % param height_tx : height of Tx
    % param height_rx : height of Rx
    '''
    if env == 'suburban':
        a1 = 0.1
        a2 = 7.5e-4
        a3 = 8
    elif env == 'urban':
        a1 = 0.3
        a2 = 5e-4
        a3 = 15
    
    delta_h = height_tx - height_rx
    # pow_factor = 2 * h_dist * math.sqrt(a1*a2/math.pi) + a1 # NOTE: Use this pow_factor if assuming PPP building dist.
    pow_factor = h_dist * math.sqrt(a1*a2) # NOTE: Use this pow_factor if assuming ITU-R assumptions.
    if delta_h == 0:
        p = (1 - math.exp((-(height_tx)**2) / (2*a3**2))) ** pow_factor
    else:
        if delta_h < 0:
            h1 = height_rx
            h2 = height_tx
        else:
            h1 = height_tx
            h2 = height_rx
        delta_h = abs(delta_h)
        p = (1 - (math.sqrt(2*math.pi)*a3 / delta_h) * abs(q_func(h1/a3) - q_func(h2/a3))) ** pow_factor
    return p

def sinr_lognormal_approx(h_dist, height, env='suburban'):
    '''
    To approximate the SNR from signal considering multipath fading and shadowing
    Assuming no interference due to CSMA, and fixed noise
    Inputs:
    h_dist = Horizontal Distance between Tx and Rx
    height = Height difference between Tx and Rx
    env = The operating environment (currently only suburban supported)
    '''
    # Signal properties
    P_Tx_dBm = 20 # Transmit power of 
    P_Tx = 10**(P_Tx_dBm/10) / 1000
    freq = 2.4e9 # Channel frequency (Hz)
    noise_dBm = -86
    noise = 10**(noise_dBm/10) / 1000
    if env == "suburban":
        # ENV Parameters Constants ----------------------------------
        # n_min = 2
        # n_max = 2.75
        # K_dB_min = 7.8
        # K_dB_max = 17.5
        # K_min = 10**(K_dB_min/10)
        # K_max = 10**(K_dB_max/10)
        # alpha = 11.25 # Env parameters for logarithm std dev of shadowing 
        # beta = 0.06 # Env parameters for logarithm std dev of shadowing 
        n_min = 2
        n_max = 2.75
        K_dB_min = 1.4922
        K_dB_max = 12.2272
        K_min = 10**(K_dB_min/10)
        K_max = 10**(K_dB_max/10)
        alpha = 11.1852 # Env parameters for logarithm std dev of shadowing 
        beta = 0.06 # Env parameters for logarithm std dev of shadowing 
        # -----------------------------------------------------------
    elif env == "urban":
        n_min = 1.9
        n_max = 2.7
        K_dB_min = -5
        K_dB_max = 15
        K_min = 10**(K_dB_min/10)
        K_max = 10**(K_dB_max/10)
        alpha = 10.42 # Env parameters for logarithm std dev of shadowing 
        beta = 0.05 # Env parameters for logarithm std dev of shadowing 
    # Calculate fading parameters
    PLoS = plos_calc(h_dist, 0, height, env=env)
    theta_Rx = math.atan2(height, h_dist) * 180 / math.pi # Elevation angle in degrees
    ple = (n_min - n_max) * PLoS + n_max # Path loss exponent
    sigma_phi_dB = alpha*math.exp(-beta*theta_Rx)
    sigma_phi = 10**(sigma_phi_dB/10) # Logarithmic std dev of shadowing
    K = K_min * math.exp(math.log(K_max/K_min) * PLoS**2)
    omega = 1 # Omega of NCS (Rician)
    dist = math.sqrt(h_dist**2 + height**2)
    P_Rx = friis_calc(P_Tx, freq, dist, ple)
    # Approximate L-NCS RV (which is the SNR) as lognormal
    eta = math.log(10) / 10
    mu_phi = 10*math.log10(P_Rx)
    E_phi = math.exp(eta*mu_phi + eta**2*sigma_phi**2/2) # Mean of shadowing RV
    var_phi = math.exp(2*eta*mu_phi+eta**2*sigma_phi**2)*(math.exp(eta**2*sigma_phi**2)-1) # Variance of shadowing RV
    E_chi = (special.gamma(1+1)/(1+K))*special.hyp1f1(-1,1,-K)*omega
    var_chi = (special.gamma(1+2)/(1+K)**2)*special.hyp1f1(-2,1,-K)*omega**2 - E_chi**2
    E_SNR = E_phi * E_chi / noise # Theoretical mean of SINR
    var_SNR = ((var_phi+E_phi**2)*(var_chi+E_chi**2) - E_phi**2 * E_chi**2) / noise**2
    std_dev_SNR = math.sqrt(var_SNR)
    # sigma_ln = math.sqrt(math.log(var_SNR/E_SNR**2 + 1))
    # mu_ln = math.log(E_SNR) - sigma_ln**2/2
    return E_SNR, std_dev_SNR

def normalize_data(df, columns=[], save_details_path=None):
    '''
    columns: The pandas data columns to normalize, given as a list of column names
    '''
    # Define the ranges of parametrers
    max_mean_sinr = 10*math.log10(1123) # The max mean SINR calculated at (0,60) is 1122.743643457063 (linear)
    max_std_dev_sinr = 10*math.log10(466) # The max std dev SINR calculated at (0,60) is 465.2159856885714 (linear)
    min_mean_sinr = 10*math.log10(0.2) # The min mean SINR calculated at (1200,60) is 0.2251212887895188 (linear)
    min_std_dev_sinr = 10*math.log10(0.7) # The min std dev SINR calculated at (1200,300) is 0.7160093126585219 (linear)
    max_height = 300
    min_height = 60
    max_h_dist = 1200
    min_h_dist = 0
    max_mcs = 7
    min_mcs = 0

    # Normalize data (Min Max Normalization between [-1,1])
    if "Height" in columns:
        df["Height"] = df["Height"].apply(lambda x: 2*(x-min_height)/(max_height-min_height) - 1)
    if "U2G_H_Dist" in columns:
        df["U2G_H_Dist"] = df["U2G_H_Dist"].apply(lambda x: 2*(x-min_h_dist)/(max_h_dist-min_h_dist) - 1)
    if "Mean_SINR" in columns:
        df["Mean_SINR"] = df["Mean_SINR"].apply(lambda x: 2*(10*math.log10(x)-min_mean_sinr)/(max_mean_sinr-min_mean_sinr) - 1) # Convert to dB space
    if "Std_Dev_SINR" in columns:
        df["Std_Dev_SINR"] = df["Std_Dev_SINR"].apply(lambda x: 2*(10*math.log10(x)-min_std_dev_sinr)/(max_std_dev_sinr-min_std_dev_sinr) - 1) # Convert to dB space
    if "UAV_Sending_Interval" in columns:
        df["UAV_Sending_Interval"] = df["UAV_Sending_Interval"].replace({10:-1, 20:-0.5, 40:0, 66.7: 0.5, 100:1, 1000:2})
    if "Packet_State" in columns:
        df['Packet_State'] = df['Packet_State'].replace({"Reliable":0, "QUEUE_OVERFLOW":1, "RETRY_LIMIT_REACHED":2, "Delay_Exceeded":3})
    if "Modulation" in columns:
        df['Modulation'] = df['Modulation'].replace({"BPSK":1, "QPSK":0.3333, 16:-0.3333, "QAM-16":-0.3333, "QAM16":-0.3333, 64:-1, "QAM-64":-1, "QAM64":-1})
    if "MCS" in columns:
        df["MCS"] = df["MCS"].apply(lambda x: 2*(x-min_mcs)/(max_mcs-min_mcs) - 1)

    # Record details of inputs and output for model
    if save_details_path is not None:
        f = open(os.path.join(save_details_path,"model_details.txt"), "w")
        f.write("Max Height (m): {}\n".format(max_height))
        f.write("Min Height (m): {}\n".format(min_height))
        f.write("Max H_Dist (m): {}\n".format(max_h_dist))
        f.write("Min H_Dist (m): {}\n".format(min_h_dist))
        f.write("Max Mean SINR (dB): {}\n".format(max_mean_sinr))
        f.write("Min Mean SINR (dB): {}\n".format(min_mean_sinr))
        f.write("Max Std Dev SINR (dB): {}\n".format(max_std_dev_sinr))
        f.write("Min Std Dev SINR (dB): {}\n".format(min_std_dev_sinr))
        f.write("[BPSK: 1, QPSK: 0.3333, QAM16: -0.3333, QAM64: -1]\n")
        f.write("UAV Sending Interval: [10:-1, 20:-0.5, 40:0, 100:0.5, 1000:1]\n")
        f.write("Output: ['Reliable':0, 'QUEUE_OVERFLOW':1, 'RETRY_LIMIT_REACHED':2, 'Delay_Exceeded':3]\n")
        f.close()

    return df

def get_mcs_index(df_in):
    '''
    Gets the MCS index based on modulation and bitrate column of the df_in
    '''
    df = df_in.copy()
    df["MCS"] = ''
    df.loc[(df["Modulation"] == "BPSK") & (df["Bitrate"] == 6.5), "MCS"] = 0 # MCS Index 0
    df.loc[(df["Modulation"] == "QPSK") & (df["Bitrate"] == 13.0), "MCS"] = 1 # MCS Index 0
    df.loc[(df["Modulation"] == "QPSK") & (df["Bitrate"] == 19.5), "MCS"] = 2 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM16") & (df["Bitrate"] == 26.0), "MCS"] = 3 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM16") & (df["Bitrate"] == 39.0), "MCS"] = 4 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 52.0), "MCS"] = 5 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 58.5), "MCS"] = 6 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 65.0), "MCS"] = 7 # MCS Index 0

    return df

def get_output_layer(model, layer_name):
    # From https://github.com/jacobgil/keras-cam/blob/master/model.py#L79
    # get the symbolic outputs of each "key" layer (we gave them unique names).
    layer_dict = dict([(layer.name, layer) for layer in model.layers])
    layer = layer_dict[layer_name]
    return layer

def find_nearest_value(value, array):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]

def find_nearest_index(value, array):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx

def build_nn_model_v4_wobatchnorm_noactivation():
    # For multiple output model
    # Version 4: Having only a single output layer for packet state
    inputs = Input(shape=(4,))
    base = Dense(100, activation='relu')(inputs)
    base = Dense(50, activation='relu')(base)
    base = Dense(25, activation='relu')(base)
    base = Dense(10, activation='relu')(base)
    packet_state_out = Dense(4, activation=None, name='packet_state_no_activation')(base)
    model = Model(inputs=inputs, outputs = packet_state_out)
    return model

2024-07-17 10:30:52.452459: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-07-17 10:30:52.561743: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-07-17 10:30:52.566371: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-07-17 10:30:52.566384: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if yo

## Investigate false predictions in the test dataset

### NN

In [18]:
TEST_DATASET = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_ParrotAR2/test_dataset_{}_processed/Downlink_Reliability.csv"
NN_MODELS = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_ParrotAR2/nn_ckpts/model_Downlink.round-1_split-9_0.2319.h5"
# NN_MODELS = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/nn_ckpts/model_Uplink.round-1_split-4_0.1346.h5"
# NN_MODELS = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/nn_ckpts/model_Video.round-0_split-8_0.2655.h5"
MAX_HDIST = 700 # Since the test datasets goes up to 1200m
REL_TH = 999

if REL_TH == 90:
    rel_th = 0.9
elif REL_TH == 99:
    rel_th = 0.99
elif REL_TH == 999:
    rel_th = 0.999

''' Load NN Model '''
nn_model = tf.keras.models.load_model(NN_MODELS, compile=False)
nn_model.compile(optimizer='adam', 
                loss={'packet_state': 'categorical_crossentropy'},
                metrics={'packet_state': 'accuracy'})

data_dtypes = {"Horizontal_Distance": np.float64, "Height":np.float64, "UAV_Sending_Interval": np.float64, "Modulation": 'str', "Bitrate": np.float64}
test_data_df_1 = pd.read_csv(TEST_DATASET.format(1), dtype=data_dtypes)
test_data_df_2 = pd.read_csv(TEST_DATASET.format(2), dtype=data_dtypes)
test_data_df = pd.concat([test_data_df_1, test_data_df_2], ignore_index=True)
test_data_df = test_data_df.loc[test_data_df["Horizontal_Distance"] <= MAX_HDIST]
test_data_df = get_mcs_index(test_data_df)
test_data_df["Reliability"] = (test_data_df["Num_Reliable"] / test_data_df["Num_Sent"]).values
test_data_df["Reliable_State_{}".format(REL_TH)] = test_data_df["Reliability"] >= rel_th
test_data_df = normalize_data(test_data_df, columns=["Mean_SINR", "Std_Dev_SINR", "UAV_Sending_Interval", "MCS"], save_details_path=None)
nn_prediction = nn_model.predict(test_data_df[["Mean_SINR", "Std_Dev_SINR", "UAV_Sending_Interval", "MCS"]].values)
test_data_df['NN_Predicted_Reliability'] = [prob[0] for prob in nn_prediction]
test_data_df["NN_Predicted_Reliable_State_{}".format(REL_TH)] = test_data_df['NN_Predicted_Reliability'] >= rel_th

# Get the false positive
fn_df = test_data_df.loc[(test_data_df["Reliable_State_{}".format(REL_TH)] == False) & (test_data_df["NN_Predicted_Reliable_State_{}".format(REL_TH)] == True)]
fn_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/journal_2_scripts/nn_fn2.csv")

 32/140 [=====>........................] - ETA: 0s

140/140 [==============================] - 0s 2ms/step


### Cal NN

In [ ]:
TEST_DATASET = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/test_dataset_{}_processed/Uplink_Reliability.csv"
NN_MODELS = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/nn_ckpts/model_Uplink.round-1_split-9_0.1363.h5"
CAL_T = 1.689699656925320
MAX_HDIST = 700 # Since the test datasets goes up to 1200m
REL_TH = 999

if REL_TH == 90:
    rel_th = 0.9
elif REL_TH == 99:
    rel_th = 0.99
elif REL_TH == 999:
    rel_th = 0.999

''' Load NN Model '''
nn_model = tf.keras.models.load_model(NN_MODELS, compile=False)
nn_model.compile(optimizer='adam', 
                loss={'packet_state': 'categorical_crossentropy'},
                metrics={'packet_state': 'accuracy'})
nn_model_no_act = build_nn_model_v4_wobatchnorm_noactivation() # For testing calibrated NN
nn_model_no_act.set_weights(nn_model.get_weights())

data_dtypes = {"Horizontal_Distance": np.float64, "Height":np.float64, "UAV_Sending_Interval": np.float64, "Modulation": 'str', "Bitrate": np.float64}
test_data_df_1 = pd.read_csv(TEST_DATASET.format(1), dtype=data_dtypes)
test_data_df_2 = pd.read_csv(TEST_DATASET.format(2), dtype=data_dtypes)
test_data_df = pd.concat([test_data_df_1, test_data_df_2], ignore_index=True)
test_data_df = test_data_df.loc[test_data_df["Horizontal_Distance"] <= MAX_HDIST]
test_data_df = get_mcs_index(test_data_df)
test_data_df["Reliability"] = (test_data_df["Num_Reliable"] / test_data_df["Num_Sent"]).values
test_data_df["Reliable_State_{}".format(REL_TH)] = test_data_df["Reliability"] >= rel_th
test_data_df = normalize_data(test_data_df, columns=["Mean_SINR", "Std_Dev_SINR", "UAV_Sending_Interval", "MCS"], save_details_path=None)
nn_logits = nn_model_no_act.predict(test_data_df[["Mean_SINR", "Std_Dev_SINR", "UAV_Sending_Interval", "MCS"]].values)
test_data_df["CalibNN_Specificity_Predicted_Reliability_{}".format(REL_TH)] = np.array([activations.softmax(K.constant([logits/CAL_T]), axis=-1)[0][0].numpy() for logits in nn_logits])
test_data_df["CalibNN_Specificity_Predicted_Reliable_State_{}".format(REL_TH)] = test_data_df["CalibNN_Specificity_Predicted_Reliability_999"] >= rel_th

# Get the false positive
fn_df = test_data_df.loc[(test_data_df["Reliable_State_{}".format(REL_TH)] == False) & (test_data_df["CalibNN_Specificity_Predicted_Reliability_{}".format(REL_TH)] == True)]
fn_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/journal_2_scripts/calnn_fn.csv")

### BN

In [15]:
TEST_DATASET = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/test_dataset_{}_processed/Uplink_Reliability.csv"
BN_MODELS = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/bn_cpts/djispark_reliability_bn_CPT_Uplink.csv"
# BN_MODELS = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/bn_cpts/djispark_reliability_bn_CPT_Uplink.csv"
# BN_MODELS = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/bn_cpts/djispark_reliability_bn_CPT_Video.csv"
MAX_HDIST = 700 # Since the test datasets goes up to 1200m
REL_TH = 999

if REL_TH == 90:
    rel_th = 0.9
elif REL_TH == 99:
    rel_th = 0.99
elif REL_TH == 999:
    rel_th = 0.999

''' Load NN Model '''
bn_cpt_df = pd.read_csv(BN_MODELS)
HDIST_BIN = np.arange(0, 710, 10) # For associating each hdist to its nearest value in train dataset
HEIGHT_BIN = np.arange(60, 330, 30) # For associating each height to its nearest value in train dataset

data_dtypes = {"Horizontal_Distance": np.float64, "Height":np.float64, "UAV_Sending_Interval": np.float64, "Modulation": 'str', "Bitrate": np.float64}
test_data_df_1 = pd.read_csv(TEST_DATASET.format(1), dtype=data_dtypes)
test_data_df_2 = pd.read_csv(TEST_DATASET.format(2), dtype=data_dtypes)
test_data_df = pd.concat([test_data_df_1, test_data_df_2], ignore_index=True)
test_data_df = test_data_df.loc[test_data_df["Horizontal_Distance"] <= MAX_HDIST]
test_data_df = get_mcs_index(test_data_df)
test_data_df["Reliability"] = (test_data_df["Num_Reliable"] / test_data_df["Num_Sent"]).values
test_data_df["Reliable_State_{}".format(REL_TH)] = test_data_df["Reliability"] >= rel_th

# Associating each horizontal distance and height with the closest values in training dataset:
test_data_df["Horizontal_Distance_Class"] = pd.cut(test_data_df["Horizontal_Distance"], bins=HDIST_BIN, right=False, include_lowest=True, labels=np.arange(0, len(HDIST_BIN)-1))
# test_data_df["Horizontal_Distance_Class"] = test_data_df["Horizontal_Distance"].apply(find_nearest_index, args=([HDIST_BIN]))
test_data_df["Height_Class"] = pd.cut(test_data_df["Height"], bins=HEIGHT_BIN, right=False, include_lowest=True, labels=np.arange(0, len(HEIGHT_BIN)-1))
# test_data_df["Height_Class"] = test_data_df["Height"].apply(find_nearest_index, args=([HEIGHT_BIN]))
test_data_df["UAV_Sending_Interval_Class"] = test_data_df["UAV_Sending_Interval"].replace({10.0:0, 20.0:1, 66.7:2, 100.0:3}) # Change sending interval categorial to numeric
test_data_df["Reliability_Class"] = pd.cut(test_data_df["Reliability"], bins=[-0.1,0.5,0.7,0.9,1], labels=["Low", "ModeratelyLow", "ModeratelyHigh", "High"])
predicted_reliability = [] # To store reliability predictions
for row in test_data_df.itertuples():
    bn_predictions = bn_cpt_df.loc[(bn_cpt_df["Horizontal_Distance_Class"]==row.Horizontal_Distance_Class) & (bn_cpt_df["Height_Class"]==row.Height_Class) &
                                    (bn_cpt_df["UAV_Sending_Interval_Class"]==row.UAV_Sending_Interval_Class) & (bn_cpt_df["MCS"]==row.MCS)]
    # print(bn_predictions)
    try:
        predicted_reliability.append(bn_predictions["Reliability"].values[0])
    except:
        print(row)
        print(bn_predictions)
        break
test_data_df['BN_Predicted_Reliability'] = predicted_reliability
test_data_df["BN_Predicted_Reliable_State_{}".format(REL_TH)] = test_data_df['BN_Predicted_Reliability'] >= rel_th

# Get the false positive
fn_df = test_data_df.loc[(test_data_df["Reliable_State_{}".format(REL_TH)] == False) & (test_data_df["BN_Predicted_Reliable_State_{}".format(REL_TH)] == True)]
fn_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/journal_2_scripts/bn_fn.csv")

## Calculate Specificity of Custom Model

### NN

In [6]:
TEST_DATASET = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_ParrotAR2/test_dataset_{}_processed/Uplink_Reliability.csv"
NN_MODELS = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_ParrotAR2/nn_ckpts_retrain/model_Uplink.round-0_split-2_0.1278.h5"
MAX_HDIST = 700 # Since the test datasets goes up to 1200m

''' Load NN Model '''
nn_model = tf.keras.models.load_model(NN_MODELS, compile=False)
nn_model.compile(optimizer='adam', 
                loss={'packet_state': 'categorical_crossentropy'},
                metrics={'packet_state': 'accuracy'})

data_dtypes = {"Horizontal_Distance": np.float64, "Height":np.float64, "UAV_Sending_Interval": np.float64, "Modulation": 'str', "Bitrate": np.float64}
test_data_df_1 = pd.read_csv(TEST_DATASET.format(1), dtype=data_dtypes)
test_data_df_2 = pd.read_csv(TEST_DATASET.format(2), dtype=data_dtypes)
test_data_df = pd.concat([test_data_df_1, test_data_df_2], ignore_index=True)
test_data_df = test_data_df.loc[test_data_df["Horizontal_Distance"] <= MAX_HDIST]
test_data_df = get_mcs_index(test_data_df)
test_data_df["Reliability"] = (test_data_df["Num_Reliable"] / test_data_df["Num_Sent"]).values
test_data_df = normalize_data(test_data_df, columns=["Mean_SINR", "Std_Dev_SINR", "UAV_Sending_Interval", "MCS"], save_details_path=None)
nn_prediction = nn_model.predict(test_data_df[["Mean_SINR", "Std_Dev_SINR", "UAV_Sending_Interval", "MCS"]].values)
test_data_df['NN_Predicted_Reliability'] = [prob[0] for prob in nn_prediction]

# Get the reliability level classifications
test_data_df["Reliable_State_90"] = test_data_df["Reliability"] >= 0.9
test_data_df["NN_Predicted_Reliable_State_90"] = test_data_df['NN_Predicted_Reliability'] >= 0.9
test_data_df["Reliable_State_99"] = test_data_df["Reliability"] >= 0.99
test_data_df["NN_Predicted_Reliable_State_99"] = test_data_df['NN_Predicted_Reliability'] >= 0.99
test_data_df["Reliable_State_999"] = test_data_df["Reliability"] >= 0.999
test_data_df["NN_Predicted_Reliable_State_999"] = test_data_df['NN_Predicted_Reliability'] >= 0.999

# Get specificity
specificity_90 = recall_score(test_data_df["Reliable_State_90"], test_data_df["NN_Predicted_Reliable_State_90"], pos_label=False, average='binary')
specificity_99 = recall_score(test_data_df["Reliable_State_99"], test_data_df["NN_Predicted_Reliable_State_99"], pos_label=False, average='binary')
specificity_999 = recall_score(test_data_df["Reliable_State_999"], test_data_df["NN_Predicted_Reliable_State_999"], pos_label=False, average='binary')
print(specificity_90, specificity_99, specificity_999)

# Get sensitivity
sensitivity_90 = recall_score(test_data_df["Reliable_State_90"], test_data_df["NN_Predicted_Reliable_State_90"], pos_label=True, average='binary')
sensitivity_99 = recall_score(test_data_df["Reliable_State_99"], test_data_df["NN_Predicted_Reliable_State_99"], pos_label=True, average='binary')
sensitivity_999 = recall_score(test_data_df["Reliable_State_999"], test_data_df["NN_Predicted_Reliable_State_999"], pos_label=True, average='binary')
print(sensitivity_90, sensitivity_99, sensitivity_999)

124/140 [=========================>....] - ETA: 0s

140/140 [==============================] - 0s 1ms/step
0.9994435169727324 0.9930629669156884 0.9963730569948187
0.9785553047404063 0.8989071038251366 0.9387096774193548
